In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("datos_raw.csv")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


# ----------------------------
# 0. Normalizar nombres de columnas
# ----------------------------
def limpiar_nombre_columna(c):
    c = str(c)
    c = c.replace("\u200b", "")
    c = c.replace("\xa0", " ")
    c = re.sub(r"\s+", " ", c)
    return c.strip()

df.columns = [limpiar_nombre_columna(c) for c in df.columns]


# ----------------------------
# 1. Normalizar provincia
# ----------------------------
def normalizar_provincia(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    equivalencias = {
        "Coruña, La": "La Coruña",
        "Palmas, Las": "Las Palmas",
        "Rioja, La": "La Rioja",
        "Baleares": "Islas Baleares",
        "Gerona": "Girona",
        "Lérida": "Lleida",
        "Orense": "Ourense",
        "Álava": "Araba/Álava",
        "Guipúzcoa": "Gipuzkoa",
        "Vizcaya": "Bizkaia"
    }

    return equivalencias.get(x, x)

df["Provincia"] = df["Provincia"].astype(str).str.strip().apply(normalizar_provincia)

filas_basura = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)", "-", "nan"]
df = df[~df["Provincia"].isin(filas_basura)].copy()


# ----------------------------
# 2. Eliminar columnas basura
# ----------------------------
cols_drop = [
    c for c in df.columns
    if c.startswith("Unnamed")
    or c.startswith("Escudo")
    or c.startswith("Mapa")
    or c.startswith("Posición")
    or c.startswith("Puesto")
    or c.startswith("Provincia.")
]

df = df.drop(columns=cols_drop, errors="ignore")


# ----------------------------
# 3. Unificar columnas equivalentes
# ----------------------------
def unificar_columnas(df, nombre_base):
    cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
    if not cols:
        return df

    df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]
    cols_a_borrar = [c for c in cols if c != nombre_base]
    df = df.drop(columns=cols_a_borrar, errors="ignore")
    return df


bases_a_unificar = [
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Población",
    "Presupuesto (€)",
    "Densidad (hab./km²)",
    "Superficie (km²)",
    "Código postal",
    "Código Ministerio del Interior",
    "Órgano de gobierno y administración",
    "Sede (ciudad)",
    "Extranjeros totales",
    "% de extranjeros",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie",
    "Longitud de costa (km)[2]​[3]​",
    "Superficie (km²)[2]​"
]

for base in bases_a_unificar:
    df = unificar_columnas(df, base)


# ----------------------------
# 4. Consolidar duplicados por provincia
# ----------------------------
def first_non_null(series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else np.nan

df = df.groupby("Provincia", as_index=False).agg(first_non_null)


# ----------------------------
# 5. Limpieza de texto
# ----------------------------
def limpiar_texto(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = re.sub(r"\[\d+\]", "", x)
    x = x.replace("\xa0", " ")
    x = x.replace("\u200b", "")
    x = x.strip()

    if x in ["", "-", "—", "nan", "None"]:
        return np.nan

    return x

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)


# ----------------------------
# 6. Funciones numéricas
# ----------------------------
def numero_entero_es(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer)):
        return int(x)

    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return np.nan
        return int(round(x))

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")
    s = s.replace("%", "")

    if s.count(".") > 1:
        s = s.replace(".", "")
        return pd.to_numeric(s, errors="coerce")

    if s.count(".") == 1 and "," not in s:
        izq, der = s.split(".")
        if der.isdigit() and len(der) == 3:
            s = izq + der
            return pd.to_numeric(s, errors="coerce")

    val = pd.to_numeric(s, errors="coerce")
    if pd.notna(val):
        return int(round(val))

    return np.nan


def numero_decimal_es(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer, float, np.floating)):
        return float(x)

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")
    s = s.replace("%", "")

    if "," in s and "." in s:
        s = s.replace(".", "")
        s = s.replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")

    return pd.to_numeric(s, errors="coerce")


def porcentaje_extranjeros_corregido(x):
    """
    En esta tabla concreta el scraper deja:
    74 -> 7.4
    218 -> 21.8
    230 -> 23.0
    """
    val = numero_decimal_es(x)
    if pd.isna(val):
        return np.nan
    return val / 10


# ----------------------------
# 7. Convertir columnas por tipo
# ----------------------------
cols_enteras = [
    "Código postal",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Presupuesto (€)",
    "Extranjeros totales",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023"
]

for col in cols_enteras:
    if col in df.columns:
        df[col] = df[col].apply(numero_entero_es)

cols_decimales = [
    "Densidad (hab./km²)",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie"
]

for col in cols_decimales:
    if col in df.columns:
        df[col] = df[col].apply(numero_decimal_es)

if "% de extranjeros" in df.columns:
    df["% de extranjeros"] = df["% de extranjeros"].apply(porcentaje_extranjeros_corregido)


# ----------------------------
# 8. Rellenar superficie auxiliar
# ----------------------------
col_aux = [c for c in df.columns if c.startswith("Superficie (km²)[2]")]
if col_aux:
    aux = col_aux[0]
    df[aux] = df[aux].apply(numero_entero_es)

    if "Superficie (km²)" not in df.columns:
        df["Superficie (km²)"] = np.nan

    df["Superficie (km²)"] = df["Superficie (km²)"].fillna(df[aux])
    df = df.drop(columns=[aux], errors="ignore")


# ----------------------------
# 9. Normalizar comunidades autónomas
# ----------------------------
def normalizar_ccaa(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    mapa = {
        "Principado de Asturias": "Asturias",
        "Comunidad Foral de Navarra": "Navarra"
    }

    return mapa.get(x, x)

if "Comunidad autónoma" in df.columns:
    df["Comunidad autónoma"] = df["Comunidad autónoma"].apply(normalizar_ccaa)

# ----------------------------
# 10. Renombramos las columnas
# ----------------------------   
lst_pib = ['2016', '2017', '2018', '2019','2020', '2021', '2022', '2023']
dct_col = {col : 'PIB_'+col for col in lst_pib}
df.rename(columns=dct_col, inplace=True)

# ----------------------------
# 10. Reordenar columnas
# ----------------------------
columnas_finales = [
    "Provincia",
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Código postal",
    "Código Ministerio del Interior",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Densidad (hab./km²)",
    "Presupuesto (€)",
    "Extranjeros totales",
    "% de extranjeros",
    "Porcentaje población",
    "Porcentaje superficie",
    "PIB_2016",
    "PIB_2017",
    "PIB_2018",
    "PIB_2019",
    "PIB_2020",
    "PIB_2021",
    "PIB_2022",
    "PIB_2023",
    "TCAC 2016-23",
    "Órgano de gobierno y administración",
    "Sede (ciudad)"
]

columnas_finales = [c for c in columnas_finales if c in df.columns]
df = df[columnas_finales]


# ----------------------------
# 11. Guardar
# ----------------------------
df.to_csv("datos_finales.csv", index=False)

print(df.head())
print("\nShape:", df.shape)
print("\nColumnas:", df.columns.tolist())
print("\nComprobación:")
print(df[["Provincia", "% de extranjeros", "Porcentaje población", "Porcentaje superficie"]].head(10))

     Provincia    Comunidad autónoma   Capital Ciudad más poblada*  \
0     Albacete    Castilla-La Mancha  Albacete            Albacete   
1     Alicante  Comunidad Valenciana  Alicante            Alicante   
2      Almería             Andalucía   Almería             Almería   
3  Araba/Álava            País Vasco   Vitoria             Vitoria   
4     Asturias              Asturias    Oviedo               Gijón   

   Código postal Código Ministerio del Interior  Superficie (km²)  Población  \
0              2                             AB           14924.0     390751   
1              3                              A            5816.0    2033566   
2              4                             AL            8774.0     770554   
3              1                             VI            3037.0     341961   
4             33                              O           10603.0    1015128   

   Densidad (hab./km²)  Presupuesto (€)  Extranjeros totales  \
0                26.18      179014

In [3]:
df_limpio = pd.read_csv("datos_finales.csv")
df_limpio

,Provincia,Comunidad autónoma,Capital,Ciudad más poblada*,Código postal,Código Ministerio del Interior,Superficie (km²),Población,Densidad (hab./km²),Presupuesto (€),Extranjeros totales,% de extranjeros,Porcentaje población,Porcentaje superficie,PIB_2016,PIB_2017,PIB_2018,PIB_2019,PIB_2020,PIB_2021,PIB_2022,PIB_2023,TCAC 2016-23,Órgano de gobierno y administración,Sede (ciudad)
0,Albacete,Castilla-La Mancha,Albacete,Albacete,2,AB,14924.0,390751,26.18,1.790145e+08,32439,7.4,0.80,2.950,7570409,7992903,8285269,8627212,8010434,8853382,9526877,10285196,35.86,Diputación de Albacete,Palacio Provincial de Albacete (Albacete)
1,Alicante,Comunidad Valenciana,Alicante,Alicante,3,A,5816.0,2033566,349.59,3.138712e+08,456257,21.8,4.14,1.150,33899084,35422930,36463569,37807882,34503374,38470028,41939432,46472807,37.09,Diputación de Alicante,Palacio Provincial de Alicante (Alicante)
2,Almería,Andalucía,Almería,Almería,4,AL,8774.0,770554,87.81,2.257500e+08,168886,17.3,1.57,1.730,14185706,15159388,15282092,16003582,14764342,14861150,16494994,18250223,28.65,Diputación de Almería,Edificio de la Diputación Provincial de Almerí...
3,Araba/Álava,País Vasco,Vitoria,Vitoria,1,VI,3037.0,341961,112.60,4.346069e+08,36831,9.1,0.70,0.600,11960737,12336129,12721159,12797528,11764228,11958641,12838267,14090604,17.81,Diputación Foral de Álava,Palacio de la Diputación Foral de Álava (Vitoria)
4,Asturias,Asturias,Oviedo,Gijón,33,O,10603.0,1015128,95.73,2.919000e+08,58387,12.5,2.07,2.100,21675441,22538267,23185680,23666618,21376392,23617641,26690102,28350371,30.79,No tiene[1]​,No tiene[1]​
5,Badajoz,Extremadura,Badajoz,Badajoz,6,BA,21766.0,665155,30.56,1.225658e+08,25469,6.6,1.35,4.300,11776505,12419839,12609609,12838980,11979224,12729424,13644123,14927003,26.75,Diputación de Badajoz,NaN
6,Barcelona,Cataluña,Barcelona,Barcelona,8,B,7733.0,5959941,771.21,3.595900e+09,1015499,22.3,12.13,1.530,158516150,166618005,173436258,180560179,161885097,173917445,191251096,209563285,32.20,Diputación de Barcelona,Casa Serra (Barcelona)
7,Bizkaia,País Vasco,Bilbao,Bilbao,48,BI,2217.0,1167233,526.49,6.635000e+08,105893,14.6,2.38,0.440,33686330,34314272,35599877,36737656,33016596,35910195,41009633,44229902,31.30,Diputación Foral de Vizcaya,Palacio de la Diputación Foral de Vizcaya (Bil...
8,Burgos,Castilla y León,Burgos,Burgos,9,BU,14292.0,362663,25.38,2.200000e+08,35158,8.2,0.74,2.820,9572090,10054308,10494196,10628487,9620320,10224118,10709841,11303996,18.09,Diputación de Burgos,NaN
9,Cantabria,Cantabria,Santander,Santander,39,S,5325.0,593623,111.56,2.236000e+08,44242,9.8,1.21,1.050,12833899,13318023,13843935,14274351,12991241,14206538,15557155,16740663,29.93,No tiene[4]​,No tiene[4]​
